In [1]:
import sys
import os

path_to_scripts = os.path.join('..', '..', 'python_scripts')
sys.path.append(path_to_scripts)

%load_ext autoreload

In [2]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features, engineer_features
from baseline_model import baseline_model_xgb, xgb_train_preproc, evaluate_trained_model, pred_preproc_xgb, xgb_prediction


%matplotlib inline

# Function checking 

In [3]:
# raw table
df_raw = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# display(df_raw.head())
# df_raw.columns

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [4]:
# train preproc function
df_pre = xgb_train_preproc(df_raw, add_year_lag=True)
df_pre
# success working!

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,TotalOutput-MW,carbon_intensity_gCO2_kWh,status,lag_48,lag_336,lag_17520,hour,day_of_week,month,is_weekend
17520,2018-09-12 00:00:00,16.5,17.1,21.2,100,0.0,0.0,0.0,1019.2,0.0,...,20739.832,150.0,okay,122.0,244.0,142.0,0,2,9,0
17521,2018-09-12 00:30:00,16.5,17.1,21.2,100,0.0,0.0,0.0,1019.2,0.0,...,20600.825,147.0,okay,124.0,252.0,140.0,0,2,9,0
17522,2018-09-12 01:00:00,16.1,15.7,15.1,100,0.0,0.0,0.0,1018.8,0.0,...,20498.196,152.0,okay,128.0,248.0,139.0,1,2,9,0
17523,2018-09-12 01:30:00,16.1,15.7,15.1,100,0.0,0.0,0.0,1018.8,0.0,...,20427.333,150.0,okay,127.0,247.0,137.0,1,2,9,0
17524,2018-09-12 02:00:00,15.9,16.1,13.3,100,0.0,0.0,0.0,1019.4,0.0,...,20070.633,146.0,okay,123.0,252.0,132.0,2,2,9,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148939,2026-03-11 21:30:00,7.6,26.0,29.9,50,0.0,0.0,0.0,1018.0,0.0,...,31433.495,87.0,okay,73.0,119.0,181.0,21,2,3,0
148940,2026-03-11 22:00:00,7.3,26.4,30.2,86,0.0,0.0,0.0,1018.3,0.0,...,31148.429,77.0,okay,65.0,100.0,168.0,22,2,3,0
148941,2026-03-11 22:30:00,7.3,26.4,30.2,86,0.0,0.0,0.0,1018.3,0.0,...,30511.622,78.0,okay,64.0,83.0,149.0,22,2,3,0
148942,2026-03-11 23:00:00,7.0,24.8,29.9,70,0.0,0.0,0.0,1018.3,0.0,...,30607.883,78.0,okay,55.0,76.0,136.0,23,2,3,0


In [ ]:
# checking train_preproc
print(df_raw.shape)
print(df_pre.shape)
print(df_pre.columns.tolist())
print(df_pre.isna().sum().sum())
# success working!

In [ ]:
# checking model function
model, X_train, X_test, y_train, y_test = baseline_model_xgb()
# success working!

In [ ]:
# evaluate trained model
metrics = evaluate_trained_model(model, X_test, y_test)
print(metrics)

In [ ]:
# pred_preproc using X_test as df_new to check fucntion is working
#takes model ouput X_train
feature_cols = X_train.columns.tolist()

X_new = pred_preproc_xgb(df_new=X_test.copy(),
                     feature_cols=feature_cols
                     )

print(X_new.shape)
print(X_new.columns.tolist() == feature_cols)
print(X_new.isna().sum().sum())

In [ ]:
# checking prediction fucntion
prediction = xgb_prediction(
            model=model,
            df_new=X_test.copy(),
            feature_cols=feature_cols
            )

print(type(prediction))
print(len(prediction))
print(prediction[:5])

## CVGridsearch for params

In [ ]:
# BASELINE
#
# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [5]:
# model init and search
xgb = XGBRegressor(
    random_state=42,
    tree_method='hist',
    n_jobs=1
)

param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    verbose=2,
    n_jobs=-1
)


In [6]:
# loading data
df = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# df = xgb_train_preproc(df)

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


In [7]:
# preproc
# defining X and target
# keeping dattime to help plotting
# splitting temporally bc multiple years

df = xgb_train_preproc(df)
target_col = 'carbon_intensity_gCO2_kWh'

# sort by datetime and reset index ooooo
df = df.sort_values('datetime').reset_index(drop=True)

X = df.drop(columns=[target_col, 'datetime'], errors='ignore')
y = df[target_col]

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df.drop(columns=[target_col, 'datetime'], errors='ignore')
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, 'datetime'], errors='ignore')
y_test = test_df[target_col]

# keep only num cola to make xgboost happy
feature_cols = X_train.select_dtypes(include='number').columns.tolist()

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

In [ ]:
# simple grid search: fit and best and pred

# grid.fit(X_train, y_train)
# best_model = grid.best_estimator_
# y_pred = best_model.predict(X_test)

In [ ]:
# inspect simple
# best_model = grid.best_estimator_
# print(best_model)

# best_params = pd.DataFrame(list(grid.best_params_.items()),columns=['parameter', 'value'])
# print(best_params)


In [ ]:
# improved grid for grid search


# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [8]:
# bigboy params_grid

param_grid_bb = {
    'n_estimators': np.arange(200, 2200, 200),
    'learning_rate': np.arange(0.01, 0.11, 0.01),
    'max_depth': np.arange(3, 11, 1),
    'min_child_weight': np.arange(1, 11, 1),
    'subsample': np.arange(0.6, 1.01, 0.1),
    'colsample_bytree': np.arange(0.6, 1.01, 0.1),
    'gamma': np.arange(0, 1.1, 0.1),
    'reg_alpha': [0, 0.01, 0.05, 0.1, 0.5, 1.0],
    'reg_lambda': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}


scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

In [ ]:
# grissearch cross val

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid_bb,
    n_iter=100,
    scoring= scoring,
    refit = 'MAE',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_


Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.4s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.4s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_child_weight=1, n_estimators=1000, reg_alpha=0.5, reg_lambda=10.0, subsample=0.7999999999999999; total time=   4.6s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.1, learning_rate=0.07, max_depth=4, min_chi

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END colsample_bytree=0.9999999999999999, gamma=0.9, learning_rate=0.03, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=5.0, subsample=0.8999999999999999; total time=  58.5s
[CV] END colsample_bytree=0.9999999999999999, gamma=0.9, learning_rate=0.03, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=5.0, subsample=0.8999999999999999; total time=  59.0s
[CV] END colsample_bytree=0.6, gamma=0.9, learning_rate=0.07, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=0.5, subsample=0.7999999999999999; total time=  37.6s
[CV] END colsample_bytree=0.6, gamma=0.9, learning_rate=0.07, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=0.5, subsample=0.7999999999999999; total time=  37.9s
[CV] END colsample_bytree=0.6, gamma=0.9, learning_rate=0.07, max_depth=10, min_child_weight=2, n_estimators=1800, reg_alpha=0.5, reg_lambda=0.5, subsample=0.7999999999999999; total time=  3

In [ ]:
# inspect simple
best_model = random_search.best_estimator_
print(best_model)

best_params = pd.DataFrame(list(random_search.best_params_.items()),columns=['parameter', 'value'])
print(best_params)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.6), device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=np.float64(0.7000000000000001), grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=np.float64(0.07), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=np.int64(3), max_leaves=None,
             min_child_weight=np.int64(9), missing=nan,
             monotone_constraints=None, multi_strategy=None,
             n_estimators=np.int64(800), n_jobs=1, num_parallel_tree=None, ...)
          parameter   value
0         subsample    1.00
1        reg_lambda    1.00
2         reg_alpha    0.05
3      n_estimators  800.00
4  min_child_we

array([167.0354 , 161.00298, 143.69601, ...,  75.88854,  71.62744,
        77.94918], shape=(28690,), dtype=float32)

In [ ]:


y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

metrics = pd.DataFrame({
    "dataset": ["train", "test"],
    "MAE": [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_train, y_train_pred)),
        np.sqrt(mean_squared_error(y_test, y_test_pred))
    ],
    "R2": [
        r2_score(y_train, y_train_pred),
        r2_score(y_test, y_test_pred)
    ],
    "MaxError": [
        max_error(y_train, y_train_pred),
        max_error(y_test, y_test_pred)
    ]
})

print(metrics)

  dataset       MAE       RMSE        R2    MaxError
0   train  9.068102  12.084325  0.974043  186.648117
1    test  9.939934  13.164795  0.949445   74.926697


In [ ]:
# predicting
# y_pred = pipeline.predict(X_test)

In [ ]:
# fit with girdsearch cv
